# 01 — How the 81 superconductor features are built

This notebook walks through the Python port of the features from
K. Hamidieh, *A data-driven statistical model for predicting the critical temperature of a superconductor*
(Comput. Mater. Sci. 154, 2018, [arXiv:1803.10260](https://arxiv.org/abs/1803.10260)).

**Goal of this part of the code:** take a chemical formula (e.g. `Ba0.2La1.8Cu1O4`) and turn it into
81 numbers that describe it, using only properties of its elements. Those 81 numbers are what the
models (linear regression, XGBoost, later quantum models) are trained on.

### Where the code lives

| What | File | R original (`reference/predict_tc/main_script_production_9.R`) |
|---|---|---|
| Element property table | [`src/supercon/elements.py`](../src/supercon/elements.py), data in [`src/supercon/data/hamidieh_elements.csv`](../src/supercon/data/hamidieh_elements.csv) | lines 1057–1071 (`subset_element_data`) |
| The 10 statistics of one property | `_property_features` in [`src/supercon/features.py`](../src/supercon/features.py) | lines 757–824 (`std`, `get_features`) |
| All 81 features of a material | `featurize` in [`src/supercon/features.py`](../src/supercon/features.py) | lines 834–940 (`extract`) |
| Formula string → element counts | `counts_from_formulas` in [`src/supercon/features.py`](../src/supercon/features.py) | CHNOSZ `makeup()` |
| Build the full feature file | [`scripts/01_build_features.py`](../scripts/01_build_features.py) | lines 1081–1091 (main loop) |
| Automated checks | [`tests/test_features.py`](../tests/test_features.py) | — |

The R script is not part of this repo; clone it with
`git clone https://github.com/khamidieh/predict_tc reference/predict_tc` (see the README).

We go step by step: **data → element table → proportions → the 10 statistics (by hand) → the
vectorized code → edge cases → the full dataset check.**

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from supercon import FEATURE_NAMES, counts_from_formulas, featurize
from supercon.elements import ELEMENTS, PROPERTIES, load_element_table

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 12)

## 1. The input: `unique_m.csv`

The author started from the NIMS SuperCon database, cleaned it (paper Section 2.2) and published the
result on the [UCI repository](https://archive.ics.uci.edu/dataset/464/superconductivty+data) as two files:

* `unique_m.csv` — one row per material: **86 columns with the count of each element** (H … Rn),
  plus `critical_temp` (K) and the formula string `material`;
* `train.csv` — the 81 features the author computed from those rows, plus `critical_temp`.

We start from `unique_m.csv` and try to rebuild `train.csv` ourselves.

In [2]:
unique_m = pd.read_csv(RAW / "unique_m.csv")
print(unique_m.shape)

row = unique_m.iloc[0]
print("material:", row["material"], "| Tc =", row["critical_temp"], "K")
el = row[ELEMENTS].astype(float)
el[el > 0]  # only the non-zero element columns

(21263, 88)
material: Ba0.2La1.8Cu1O4 | Tc = 29.0 K


O     4.0
Cu    1.0
Ba    0.2
La    1.8
Name: 0, dtype: float64

Most of the 86 columns are zero; a material only has a handful of elements. Row 0 is
`Ba0.2La1.8Cu1O4`: 0.2 Ba, 1.8 La, 1 Cu and 4 O.

## 2. The element property table — [`src/supercon/elements.py`](../src/supercon/elements.py)

Each feature is a statistic of one **elemental property** over the elements in the material.
The paper uses 8 properties (Table 1 of the paper). The author took them from Mathematica's `ElementData`,
which is not freely available, so we use the author's exact table: the R object `subset_element_data`,
extracted from the author's `tc.RData` file and bundled in
[`src/supercon/data/hamidieh_elements.csv`](../src/supercon/data/hamidieh_elements.csv).

`PROPERTIES` maps each property to the short name used in the feature columns:

In [3]:
PROPERTIES

{'AtomicMass': 'atomic_mass',
 'FirstIonizationEnergy': 'fie',
 'AtomicRadius': 'atomic_radius',
 'Density': 'Density',
 'ElectronAffinity': 'ElectronAffinity',
 'FusionHeat': 'FusionHeat',
 'ThermalConductivity': 'ThermalConductivity',
 'Valence': 'Valence'}

In [4]:
table = load_element_table()   # 86 elements x 8 properties
table.loc[["H", "O", "Cu", "Zr", "Ba", "La", "Ce", "Re"]]

,AtomicMass,FirstIonizationEnergy,AtomicRadius,Density,ElectronAffinity,FusionHeat,ThermalConductivity,Valence
Element,,,,,,,,
H,1.00794,1311.3,53,0.0899,74.30,0.558,0.18050,1
O,15.99940,1313.1,48,1.4290,142.50,0.222,0.02658,2
Cu,63.54600,745.0,145,8960.0000,119.90,13.100,400.00000,2
Zr,91.22400,659.7,206,6511.0000,42.60,21.000,23.00000,4
Ba,137.32700,502.5,253,3510.0000,15.45,8.000,18.00000,2
La,138.90547,541.1,195,6146.0000,49.50,6.300,13.00000,3
Ce,140.11600,540.1,185,6689.0000,51.50,5.500,11.00000,4
Re,186.20700,759.1,188,21020.0000,16.00,33.000,48.00000,7


The table already includes two adjustments the author made (R lines 1064–1071):

* **La and Ce** had no atomic radius, so they were filled with 195 and 185 pm;
* **1.5 was added to every electron affinity**, because some elements (e.g. He) have 0 and the
  features take logarithms.

Two values are still missing (NaN). We come back to them in Section 6.

In [5]:
table[table.isna().any(axis=1)]

,AtomicMass,FirstIonizationEnergy,AtomicRadius,Density,ElectronAffinity,FusionHeat,ThermalConductivity,Valence
Element,,,,,,,,
Po,209.0,813.1,135,9196.0,184.8,13.0,NaN,6
At,210.0,916.3,127,NaN,271.6,6.0,2.0,7


## 3. From a formula to proportions

The only thing a feature needs from the material is **which elements it has and in what proportion**.

`counts_from_formulas` (in [`features.py`](../src/supercon/features.py)) uses
[pymatgen](https://pymatgen.org/) to parse a formula string into element counts. It plays the role of
the CHNOSZ `makeup()` function in the R code.

In [6]:
counts = counts_from_formulas(["Ba0.2La1.8Cu1O4", "MgB2", "Re6Zr1"])
counts

,Ba,La,Cu,O,Mg,B,Re,Zr
0,0.2,1.8,1.0,4.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,6.0,1.0


The proportions $p_i$ are the counts divided by the total number of atoms in the formula.
For `Re6Zr1`: $p_{Re} = 6/7$, $p_{Zr} = 1/7$.

In [7]:
p = counts.div(counts.sum(axis=1), axis=0)
p.round(4)

,Ba,La,Cu,O,Mg,B,Re,Zr
0,0.0286,0.2571,0.1429,0.5714,0.0000,0.0000,0.0000,0.0000
1,0.0000,0.0000,0.0000,0.0000,0.3333,0.6667,0.0000,0.0000
2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.8571,0.1429


Sanity check: the pymatgen parser gives exactly the same counts as the author's `unique_m.csv`
for all 21,263 materials (this is also `test_formula_parsing_matches_unique_m` in
[`tests/test_features.py`](../tests/test_features.py)). So new materials can be featurized straight
from their formula.

In [8]:
parsed = counts_from_formulas(unique_m["material"]).reindex(columns=ELEMENTS, fill_value=0.0)
print("identical counts:", np.allclose(parsed.to_numpy(), unique_m[ELEMENTS].to_numpy(), rtol=1e-12))

identical counts: True


## 4. The 10 statistics, computed by hand

This is the heart of the method (paper Section 2.3, Table 2). Take one material and one property.
Let $t_i$ be the property value of element $i$ and $p_i$ its proportion. Two more sets of weights are
derived from them:

$$w_i = \frac{t_i}{\sum_j t_j} \qquad\qquad A_i = \frac{p_i\,w_i}{\sum_j p_j\,w_j}$$

Then the 10 features are:

| Feature | Formula | Idea |
|---|---|---|
| `mean` | $\frac{1}{n}\sum t_i$ | plain average over the distinct elements |
| `wtd_mean` | $\sum p_i t_i$ | average weighted by how much of each element there is |
| `gmean` | $(\prod t_i)^{1/n}$ | geometric mean |
| `wtd_gmean` | $\prod t_i^{p_i}$ | weighted geometric mean |
| `entropy` | $-\sum w_i \ln w_i$ | how evenly the property is spread over the elements |
| `wtd_entropy` | $-\sum A_i \ln A_i$ | same, weighted by proportions |
| `range` | $\max t - \min t$ | spread |
| `wtd_range` | $\max(p_i t_i) - \min(p_i t_i)$ | spread of the weighted values |
| `std` | $\sqrt{\frac{1}{n}\sum (t_i - \text{mean})^2}$ | population standard deviation |
| `wtd_std` | $\sqrt{\sum p_i (t_i - \text{wtd\_mean})^2}$ | weighted standard deviation |

The paper's worked example is the **thermal conductivity** of Re and Zr: $t_{Re} = 48$, $t_{Zr} = 23$ W/(m·K).

> **Note on the paper:** the example is labelled "Re$_7$Zr$_1$, $T_c$ = 6.7 K", but the numbers use
> $p = 6/7, 1/7$. The dataset has no Re$_7$Zr$_1$; it has **Re$_6$Zr$_1$ with $T_c$ = 6.7 K** (row 21240),
> whose stored features are exactly the paper's Table 2. So only the name is a typo, and we use Re$_6$Zr$_1$.

Below is a deliberately simple, one-material version (no masks, no loops over materials), so each line
maps to one row of the table:

In [9]:
def ten_features_by_hand(t, p):
    t, p = np.asarray(t, float), np.asarray(p, float)
    n = len(t)
    w = t / t.sum()
    A = p * w / (p * w).sum()
    mean = t.mean()
    wtd_mean = (p * t).sum()
    return {
        "mean": mean,
        "wtd_mean": wtd_mean,
        "gmean": t.prod() ** (1 / n),
        "wtd_gmean": np.prod(t ** p),
        "entropy": -(w * np.log(w)).sum(),
        "wtd_entropy": -(A * np.log(A)).sum(),
        "range": t.max() - t.min(),
        "wtd_range": (p * t).max() - (p * t).min(),
        "std": np.sqrt(((t - mean) ** 2).mean()),
        "wtd_std": np.sqrt((p * (t - wtd_mean) ** 2).sum()),
    }

t = [48, 23]        # thermal conductivity of Re, Zr
p_re6zr1 = [6 / 7, 1 / 7]
paper_table2 = {"mean": 35.5, "wtd_mean": 44.43, "gmean": 33.23, "wtd_gmean": 43.21,
                "entropy": 0.63, "wtd_entropy": 0.26, "range": 25, "wtd_range": 37.86,
                "std": 12.5, "wtd_std": 8.75}

pd.DataFrame({"by hand": ten_features_by_hand(t, p_re6zr1), "paper Table 2": paper_table2}).round(2)

,by hand,paper Table 2
mean,35.50,35.50
wtd_mean,44.43,44.43
gmean,33.23,33.23
wtd_gmean,43.21,43.21
entropy,0.63,0.63
wtd_entropy,0.26,0.26
range,25.00,25.00
wtd_range,37.86,37.86
std,12.50,12.50
wtd_std,8.75,8.75


The hand computation reproduces the paper. Now the same thing with the package function `featurize`,
which returns all 81 features at once; we pick the 10 thermal conductivity ones:

In [10]:
x = featurize(pd.DataFrame([{"Re": 6, "Zr": 1}]))
x.filter(like="ThermalConductivity").T.round(2)

,0
mean_ThermalConductivity,35.50
wtd_mean_ThermalConductivity,44.43
gmean_ThermalConductivity,33.23
wtd_gmean_ThermalConductivity,43.21
entropy_ThermalConductivity,0.63
wtd_entropy_ThermalConductivity,0.26
range_ThermalConductivity,25.00
wtd_range_ThermalConductivity,37.86
std_ThermalConductivity,12.50
wtd_std_ThermalConductivity,8.75


Repeating these 10 statistics for each of the 8 properties gives 80 features. The 81st is
`number_of_elements`. `FEATURE_NAMES` lists them in the same order as the author's `train.csv`:

In [11]:
print(len(FEATURE_NAMES), "features")
print(FEATURE_NAMES[:12])

81 features
['number_of_elements', 'mean_atomic_mass', 'wtd_mean_atomic_mass', 'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass', 'wtd_entropy_atomic_mass', 'range_atomic_mass', 'wtd_range_atomic_mass', 'std_atomic_mass', 'wtd_std_atomic_mass', 'mean_fie']


## 5. How `featurize` does it for 21,263 materials at once

The R code loops over materials and, for each, over properties (`extract` → `get_features`).
The Python port in [`features.py`](../src/supercon/features.py) does the same maths on whole matrices
with NumPy, which takes about a second for the full dataset instead of a minute in R.

The trick is to work with the full 86-column table and a **mask** that says which elements are in each
material:

* `c` — counts, shape (materials × 86);
* `p = c / c.sum(axis=1)` — proportions;
* `mask = (c > 0) & ~isnan(t)` — "this element is in the material *and* has a value for this property".

Every sum, max and min is then taken only over the masked entries (the small helpers `msum`, `mmax`,
`mmin` inside `_property_features`). A tiny example with 3 materials and 4 elements:

In [12]:
small = pd.DataFrame(
    [{"Re": 6, "Zr": 1}, {"Mg": 1, "B": 2}, {"Ba": 0.2, "La": 1.8, "Cu": 1, "O": 4}],
    index=["Re6Zr1", "MgB2", "Ba0.2La1.8Cu1O4"],
)
cols = ["Re", "Zr", "Mg", "B", "Ba", "La", "Cu", "O"]
c = small.reindex(columns=cols).fillna(0).to_numpy(float)  # absent elements -> 0
t = table.loc[cols, "ThermalConductivity"].to_numpy()

p = c / c.sum(axis=1, keepdims=True)
mask = (c > 0) & ~np.isnan(t)

msum = lambda v: np.where(mask, v, 0.0).sum(axis=1)   # sum over the elements of each material
print("mask:\n", mask.astype(int))
print("wtd_mean =", msum(p * t))

mask:
 [[1 1 0 0 0 0 0 0]
 [0 0 1 1 0 0 0 0]
 [0 0 0 0 1 1 1 1]]
wtd_mean = [44.42857143 71.33333333 61.01518857]


In [13]:
featurize(small)[["number_of_elements", "wtd_mean_ThermalConductivity", "range_ThermalConductivity"]]

,number_of_elements,wtd_mean_ThermalConductivity,range_ThermalConductivity
Re6Zr1,2.0,44.428571,25.00000
MgB2,2.0,71.333333,133.00000
Ba0.2La1.8Cu1O4,4.0,61.015189,399.97342


The masked sum gives the same `wtd_mean` as `featurize`.

> Note: building `small` from dicts leaves **NaN** (not 0) where a material lacks an element, hence the
> `fillna(0)`. `featurize` does the same internally; this was a bug caught while writing this notebook,
> now covered by `test_rows_with_different_elements`. Each of the 10 lines in the `column_stack`
of `_property_features` is one formula of the table in Section 4.

## 6. Matching R exactly: edge cases

To reproduce `train.csv` bit-for-bit, the port copies a few behaviours of the R code that you might
not guess from the paper:

1. **Missing property values** (Po has no thermal conductivity, At has no density): the element is
   dropped *for that property only*, and **the proportions are not renormalized**. For `Po1Cu3`,
   thermal conductivity only "sees" Cu with $p = 0.75$, so `wtd_mean` is $0.75 \times t_{Cu}$, not $t_{Cu}$.
2. **Zeros under a logarithm** (He has valence 0): R gives `gmean = 0` and `entropy = NaN`
   (because $0 \cdot \ln 0$ is NaN). NumPy does the same, so we get NaN too rather than hiding it.
3. `std` is the **population** std (divide by $n$, R line 757), not the sample std.
4. `wtd_range` is $\max(p_i t_i) - \min(p_i t_i)$ (R line 812).

In [14]:
cu = table.loc["Cu", "ThermalConductivity"]
x = featurize(pd.DataFrame([{"Po": 1, "Cu": 3}])).iloc[0]
print(f"t_Cu = {cu}")
print("mean_ThermalConductivity     =", x["mean_ThermalConductivity"])
print("wtd_mean_ThermalConductivity =", x["wtd_mean_ThermalConductivity"], "(= 0.75 * t_Cu)")
print("number_of_elements           =", x["number_of_elements"], "(Po still counts)")

t_Cu = 400.0
mean_ThermalConductivity     = 400.0
wtd_mean_ThermalConductivity = 300.0 (= 0.75 * t_Cu)
number_of_elements           = 2.0 (Po still counts)


In [15]:
x = featurize(pd.DataFrame([{"He": 1, "Cu": 1}])).iloc[0]
x[["gmean_Valence", "entropy_Valence", "wtd_mean_Valence"]]

gmean_Valence       0.0
entropy_Valence     NaN
wtd_mean_Valence    1.0
Name: 0, dtype: float64

In the published dataset **none of these cases occur**: no material contains Po, At, or a noble gas.
They matter only when featurizing new formulas.

In [16]:
print({el: int((unique_m[el] > 0).sum()) for el in ["Po", "At", "He", "Ne", "Ar", "Kr", "Xe", "Rn"]})

{'Po': 0, 'At': 0, 'He': 0, 'Ne': 0, 'Ar': 0, 'Kr': 0, 'Xe': 0, 'Rn': 0}


## 7. The full check: do we reproduce the author's `train.csv`?

This is what [`scripts/01_build_features.py`](../scripts/01_build_features.py) runs, followed by the
comparison done in `test_reproduces_train_csv` ([`tests/test_features.py`](../tests/test_features.py)).

In [17]:
train = pd.read_csv(RAW / "train.csv")
ours = featurize(unique_m[ELEMENTS])

print("same Tc column, same order:", np.array_equal(unique_m["critical_temp"], train["critical_temp"]))
print("same feature names        :", list(train.columns[:81]) == FEATURE_NAMES)

diff = (ours - train[FEATURE_NAMES]).abs()
print(f"max absolute difference over {diff.size:,} values: {diff.to_numpy().max():.1e}")
diff.max().sort_values(ascending=False).head(5)

same Tc column, same order: True
same feature names        : True
max absolute difference over 1,722,303 values: 7.6e-11


wtd_gmean_Density    7.639755e-11
gmean_Density        6.912160e-11
wtd_mean_Density     4.911271e-11
std_Density          4.911271e-11
wtd_std_Density      4.911271e-11
dtype: float64

The largest difference is ~1e-10, on Density features whose values are around 10,000 — i.e. floating
point round-off. **The Python port reproduces all 21,263 × 81 values of the author's `train.csv`.**

## 8. Something to keep in mind for modelling

The author removed rows only when *both* the composition *and* $T_c$ were identical. The same material
measured in different papers therefore appears several times with different $T_c$:

In [18]:
dup = unique_m.groupby("material")["critical_temp"].agg(["count", "min", "max"])
print(f"{(dup['count'] > 1).sum():,} formulas appear more than once "
      f"({dup.loc[dup['count'] > 1, 'count'].sum():,} of {len(unique_m):,} rows)")
dup.loc[["Re6Zr1", "Re2Zr1"]]

2,261 formulas appear more than once (7,982 of 21,263 rows)


,count,min,max
material,,,
Re6Zr1,2,6.7,7.4
Re2Zr1,3,5.9,6.8


With a random train/test split, copies of the same material can land on both sides, which makes the
test error look better than it would be for a genuinely new material. In the modelling step we first
reproduce the paper's random split (to check we get ~9.5 K RMSE) and then also use a split that keeps
each formula entirely in train or in test.

## Summary

* [`elements.py`](../src/supercon/elements.py): the 86 × 8 element table the author used.
* [`features.py`](../src/supercon/features.py): `counts_from_formulas` (formula → counts), `featurize`
  (counts → 81 features), with `_property_features` holding the 10 statistics.
* The port matches the author's `train.csv` to round-off, and is checked by `uv run pytest`.
* Next: [`scripts/01_build_features.py`](../scripts/01_build_features.py) writes
  `data/processed/features.csv`, the input for the modelling notebooks.